# DCA Buy-the-Bear Monthly BTCUSDT Backtest

Buy $10 BTC every red monthly candle close. Sell 70% at every new all-time high monthly close.

**Strategy:**
- Entry: Monthly close < monthly open → buy $10 BTC at close
- Exit: Monthly close > all previous monthly closes (new ATH) → sell 70% of holdings at close
- Always buy on red candles regardless of position or recent sell

In [1]:
import pandas as pd
import requests
import time
import json
from datetime import datetime, timezone

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: '%.8f' % x)

In [2]:
# Fetch monthly BTCUSDT klines from Binance

def fetch_monthly_klines(symbol='BTCUSDT', limit=500):
    """Fetch all monthly klines from Binance spot."""
    url = 'https://api.binance.com/api/v3/klines'
    all_klines = []
    start_time = 0
    
    while True:
        params = {
            'symbol': symbol,
            'interval': '1M',
            'startTime': start_time,
            'limit': limit
        }
        resp = requests.get(url, params=params)
        data = resp.json()
        if not data or len(data) == 0:
            break
        all_klines.extend(data)
        print(f'Fetched {len(data)} months, up to {datetime.fromtimestamp(data[-1][0]/1000, tz=timezone.utc).strftime("%Y-%m")}')
        if len(data) < limit:
            break
        start_time = data[-1][0] + 1
        time.sleep(0.1)
    
    print(f'Total months fetched: {len(all_klines)}')
    return all_klines

klines = fetch_monthly_klines()

Fetched 108 months, up to 2026-07
Total months fetched: 108


In [3]:
# Convert to DataFrame

columns = [
    'timestamp', 'open', 'high', 'low', 'close', 'volume',
    'close_time', 'quote_vol', 'trades', 'taker_buy_base',
    'taker_buy_quote', 'ignore'
]

df = pd.DataFrame(klines, columns=columns)
df['date'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
for c in ['open', 'high', 'low', 'close', 'volume']:
    df[c] = df[c].astype(float)
df = df[['date', 'open', 'high', 'low', 'close', 'volume']].copy()
df = df.sort_values('date').reset_index(drop=True)
print(f'{df["date"].min().strftime("%Y-%m")} → {df["date"].max().strftime("%Y-%m")} ({len(df)} months)')
df.tail(3)

2017-08 → 2026-07 (108 months)


,date,open,high,low,close,volume
105,2026-05-01 00:00:00+00:00,76346.58000000,82850.00000000,72512.49000000,73674.39000000,425331.03039000
106,2026-06-01 00:00:00+00:00,73674.39000000,74092.00000000,58115.01000000,58624.71000000,632661.43583000
107,2026-07-01 00:00:00+00:00,58624.71000000,64700.00000000,57800.19000000,63010.00000000,142991.06385000


In [4]:
# Strategy state variables

btc_held = 0.0
usdt_reserve = 0.0  # cash from sales (external cash not tracked here)
total_invested = 0.0
highest_close = 0.0

records = []

for i, row in df.iterrows():
    close = row['close']
    month = row['date']
    action = 'nothing'
    
    # 1. Entry: buy $10 on red monthly candle (money comes from external pocket)
    if close < row['open']:
        btc_bought = 10.0 / close
        btc_held += btc_bought
        total_invested += 10.0
        # usdt_reserve does NOT decrease — the $10 is new external capital
        action = f'buy $10 @ {close:.2f} (+{btc_bought:.8f} BTC)'
    
    # 2. Update highest close before checking sell
    prev_highest = highest_close
    if close > highest_close:
        highest_close = close
    
    # 3. Exit: sell 70% at new all-time high
    #    (only if we have BTC AND this candle actually set a new ATH)
    if btc_held > 0.000001 and close > prev_highest:
        sell_btc = btc_held * 0.7
        sell_usdt = sell_btc * close
        btc_held -= sell_btc
        usdt_reserve += sell_usdt  # cash from sales stays in the portfolio
        act = f'sell 70% @ {close:.2f} (-{sell_btc:.8f} BTC, +{sell_usdt:.2f} USDT)'
        action = f'{action} | {act}' if 'buy' in action else act
    
    portfolio_value = btc_held * close + usdt_reserve
    
    records.append({
        'date': month,
        'close': close,
        'action': action,
        'btc_held': btc_held,
        'usdt_reserve': usdt_reserve,
        'total_invested': total_invested,
        'portfolio_value': portfolio_value,
    })

results = pd.DataFrame(records)
print(f'Total months: {len(results)}')
print(f'Trades with action: {len(results[results["action"] != "nothing"])}')
results.tail(5)

Total months: 108
Trades with action: 67


,date,close,action,btc_held,usdt_reserve,total_invested,portfolio_value
103,2026-03-01 00:00:00+00:00,68284.48000000,nothing,0.00069711,551.49510724,490.00000000,599.09660689
104,2026-04-01 00:00:00+00:00,76346.57000000,nothing,0.00069711,551.49510724,490.00000000,604.71673571
105,2026-05-01 00:00:00+00:00,73674.39000000,buy $10 @ 73674.39 (+0.00013573 BTC),0.00083284,541.49510724,500.00000000,602.85394383
106,2026-06-01 00:00:00+00:00,58624.71000000,buy $10 @ 58624.71 (+0.00017058 BTC),0.00100341,531.49510724,510.00000000,590.31999742
107,2026-07-01 00:00:00+00:00,63010.00000000,nothing,0.00100341,531.49510724,510.00000000,594.72026145


In [5]:
# Summary

import numpy as np

final = results.iloc[-1]
last_close = final['close']

# Track P&L and return index
results['pnl'] = results['portfolio_value'] - results['total_invested']

# Max drawdown (on portfolio_value, starting from first month with investment)
mask = results['total_invested'] > 0
eq = results.loc[mask, 'portfolio_value'].copy()
running_max = eq.cummax()
dd = pd.Series(0.0, index=results.index)
dd.loc[mask] = (running_max - eq) / running_max.replace(0, np.nan)
dd = dd.fillna(0)
max_drawdown = dd.max()

# Sharpe ratio (annualized, 0% risk-free) — monthly returns on portfolio_value
monthly_returns = results.loc[mask, 'portfolio_value'].pct_change().dropna()
sharpe = (monthly_returns.mean() / monthly_returns.std()) * np.sqrt(12) if monthly_returns.std() > 0 else 0.0

# Profit factor: sum of positive monthly returns / sum of |negative|
positive_sum = monthly_returns[monthly_returns > 0].sum()
negative_sum = monthly_returns[monthly_returns < 0].abs().sum()
profit_factor = positive_sum / negative_sum if negative_sum > 0 else float('inf')

print('='*60)
print('PORTFOLIO SUMMARY')
print('='*60)
print(f'Total invested:        {final["total_invested"]:.2f} USDT')
print(f'BTC held:              {final["btc_held"]:.8f} BTC')
print(f'BTC held value:        {final["btc_held"] * last_close:.2f} USDT (at {last_close:.2f})')
print(f'USDT reserve:          {final["usdt_reserve"]:.2f} USDT')
print(f'Portfolio value:       {final["portfolio_value"]:.2f} USDT')
print(f'Total P&L:             {final["portfolio_value"] - final["total_invested"]:.2f} USDT')
print(f'Return:                {((final["portfolio_value"] / final["total_invested"]) - 1) * 100:.2f}%')
print(f'Max drawdown:          {max_drawdown * 100:.2f}%')
print(f'Sharpe ratio:          {sharpe:.2f}')
print(f'Profit factor:         {profit_factor:.2f}')
print(f'Months:                {len(results)}')

# Count buys and sells
buys = [r for r in records if 'buy' in r['action']]
sells = [r for r in records if 'sell' in r['action']]
print(f'Buy months:            {len(buys)}')
print(f'Sell months:           {len(sells)}')

PORTFOLIO SUMMARY
Total invested:        510.00 USDT
BTC held:              0.00100341 BTC
BTC held value:        63.23 USDT (at 63010.00)
USDT reserve:          531.50 USDT
Portfolio value:       594.72 USDT
Total P&L:             84.72 USDT
Return:                16.61%
Months:                108
Buy months:            51
Sell months:           16


In [6]:
# All trades with actions

pd.set_option('display.max_rows', None)
with_actions = results[results['action'] != 'nothing'].copy()
with_actions[['date', 'close', 'action', 'btc_held', 'usdt_reserve', 'total_invested', 'portfolio_value']]

,date,close,action,btc_held,usdt_reserve,total_invested,portfolio_value
1,2017-09-01 00:00:00+00:00,4378.51000000,buy $10 @ 4378.51 (+0.00228388 BTC),0.00228388,-10.00000000,10.00000000,0.00000000
2,2017-10-01 00:00:00+00:00,6463.00000000,"sell 70% @ 6463.00 (-0.00159872 BTC, +10.33 USDT)",0.00068516,0.33251037,10.00000000,4.76072911
3,2017-11-01 00:00:00+00:00,9838.96000000,"sell 70% @ 9838.96 (-0.00047962 BTC, +4.72 USDT)",0.00020555,5.05142526,10.00000000,7.07381735
4,2017-12-01 00:00:00+00:00,13716.36000000,"sell 70% @ 13716.36 (-0.00014388 BTC, +1.97 USDT)",0.00006166,7.02499773,10.00000000,7.87081450
5,2018-01-01 00:00:00+00:00,10285.10000000,buy $10 @ 10285.10 (+0.00097228 BTC),0.00103395,-2.97500227,20.00000000,7.65922649
7,2018-03-01 00:00:00+00:00,6923.91000000,buy $10 @ 6923.91 (+0.00144427 BTC),0.00247822,-12.97500227,30.00000000,4.18394056
9,2018-05-01 00:00:00+00:00,7485.01000000,buy $10 @ 7485.01 (+0.00133600 BTC),0.00381422,-22.97500227,40.00000000,5.57446742
10,2018-06-01 00:00:00+00:00,6390.07000000,buy $10 @ 6390.07 (+0.00156493 BTC),0.00537915,-32.97500227,50.00000000,1.39812612
12,2018-08-01 00:00:00+00:00,7011.21000000,buy $10 @ 7011.21 (+0.00142629 BTC),0.00680543,-42.97500227,60.00000000,4.73932974
13,2018-09-01 00:00:00+00:00,6626.57000000,buy $10 @ 6626.57 (+0.00150908 BTC),0.00831451,-52.97500227,70.00000000,2.12168732
